# Gikomba Stall Analysis: Wanjiku's Fresh-Produce Business

**Analyst:** Boniface Vundi Kimeu  
**Client:** Wanjiku, fresh-produce stall, Gikomba Market  
**Data:** 14 days of daily sales (KES) and a stock record for five products  

Wanjiku asked four things: *How is my business doing? Is it growing? What should I restock? Where should I focus?*
This notebook answers them using **core Python only** (lists, dictionaries, functions, loops and f-strings). No pandas or NumPy.

## Dataset

The data cell below is copied exactly as provided.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics and best/worst days

**Approach:** I use `sum()`, `len()`, `max()` and `min()` for the totals and extremes. Then `list.index()` gives the position of the best and worst day.
Positions 0 to 6 are week 1 and positions 7 to 13 are week 2, so the week number is `position // 7 + 1` and the day name is `days[position]`.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_idx = daily_sales.index(highest_sales)
worst_idx = daily_sales.index(lowest_sales)

best_day, best_week = days[best_idx], best_idx // 7 + 1
worst_day, worst_week = days[worst_idx], worst_idx // 7 + 1

print(f"Total sales   : KES {total_sales:,}")
print(f"Average daily : KES {average_sales:,.2f}")
print(f"Highest day   : KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest day    : KES {lowest_sales:,} ({worst_day}, week {worst_week})")

Total sales   : KES 46,450
Average daily : KES 3,317.86
Highest day   : KES 5,600 (Sat, week 2)
Lowest day    : KES 1,950 (Tue, week 1)


## Task 2: Weekly totals and week-on-week growth

**Approach:** `week_totals()` slices the list into `sales[:7]` (week 1) and `sales[7:14]` (week 2) and returns both totals as a tuple.
I unpack the tuple and compute growth as `(week 2 - week 1) / week 1 * 100`.

In [3]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) from a list of 14 daily sales.

    Week 1 is the first seven values (slice [:7]) and week 2 is the next
    seven values (slice [7:14]).
    """
    return sum(sales[:7]), sum(sales[7:14])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total : KES {week1_total:,}")
print(f"Week 2 total : KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:+.1f}%")

Week 1 total : KES 22,050
Week 2 total : KES 24,400
Week-on-week growth: +10.7%


## Task 3: Classifying each day as Strong, Normal or Weak

**Approach:** `classify_day()` compares a day's sales to the 14-day average.
More than 20% above average (`> average * 1.2`) is **Strong**, more than 20% below (`< average * 0.8`) is **Weak**, and anything else is **Normal**.
I apply it to every day and count the results with the dictionary counting pattern: `counts[label] = counts.get(label, 0) + 1`.

In [4]:
def classify_day(amount, average):
    """Classify a day's sales relative to the average.

    Returns "Strong" if amount is more than 20% above average,
    "Weak" if it is more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    return "Normal"


day_labels = []
category_counts = {}
for day, amount in zip(days, daily_sales):
    label = classify_day(amount, average_sales)
    day_labels.append(label)
    category_counts[label] = category_counts.get(label, 0) + 1

print(f"Strong threshold: above KES {average_sales * 1.2:,.2f}")
print(f"Weak threshold  : below KES {average_sales * 0.8:,.2f}\n")
for i, (day, amount, label) in enumerate(zip(days, daily_sales, day_labels), start=1):
    print(f"Day {i:>2} ({day}): KES {amount:>6,}  -> {label}")

print("\nCounts:", category_counts)

Strong threshold: above KES 3,981.43
Weak threshold  : below KES 2,654.29

Day  1 (Mon): KES  2,300  -> Weak
Day  2 (Tue): KES  1,950  -> Weak
Day  3 (Wed): KES  2,800  -> Normal
Day  4 (Thu): KES  2,100  -> Weak
Day  5 (Fri): KES  3,600  -> Normal
Day  6 (Sat): KES  5,200  -> Strong
Day  7 (Sun): KES  4,100  -> Strong
Day  8 (Mon): KES  2,500  -> Weak
Day  9 (Tue): KES  2,200  -> Weak
Day 10 (Wed): KES  3,100  -> Normal
Day 11 (Thu): KES  2,400  -> Weak
Day 12 (Fri): KES  3,900  -> Normal
Day 13 (Sat): KES  5,600  -> Strong
Day 14 (Sun): KES  4,700  -> Strong

Counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus weekends

**Approach:** I loop over the sales with their day names. `Sat` and `Sun` go into a weekend list and the other days go into a weekday list.
Then I compare the two averages and express the gap as a percentage of the weekday average.

In [5]:
weekday_sales = []
weekend_sales = []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_uplift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}  ({len(weekday_sales)} days)")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}  ({len(weekend_sales)} days)")
print(f"Weekends sell {weekend_uplift_pct:.1f}% more per day than weekdays")

Weekday average (Mon-Fri): KES 2,685.00  (10 days)
Weekend average (Sat-Sun): KES 4,900.00  (4 days)
Weekends sell 82.5% more per day than weekdays


## Task 5: Reorder list and gross margins

**Approach:** `products_to_reorder()` loops through `stock.items()` and collects every product whose `qty` is **at or below** its `reorder` level (`<=`).
Gross margin % is `(price - cost) / price * 100`, calculated for each product in a second loop. I store the results in a dictionary so the report can reuse them.

In [6]:
def products_to_reorder(stock_data):
    """Return a list of product names whose quantity is at or below the reorder level."""
    low_stock = []
    for product, info in stock_data.items():
        if info["qty"] <= info["reorder"]:
            low_stock.append(product)
    return low_stock


reorder_list = products_to_reorder(stock)

margins = {}
for product, info in stock.items():
    margins[product] = (info["price"] - info["cost"]) / info["price"] * 100

print("Products to reorder:", reorder_list)
print()
print(f"{'Product':<10} {'Stock':>6} {'Reorder':>8} {'Margin':>8}")
for product, info in stock.items():
    print(f"{product:<10} {info['qty']:>4}kg {info['reorder']:>6}kg {margins[product]:>7.1f}%")

Products to reorder: ['Tomatoes', 'Sukuma']

Product     Stock  Reorder   Margin
Tomatoes     18kg     25kg    33.3%
Onions       40kg     20kg    30.0%
Sukuma        9kg     15kg    40.0%
Potatoes     55kg     30kg    33.3%
Avocado      12kg     10kg    42.9%


## Task 6: Stall Performance Report

**Approach:** I reuse the variables from Tasks 1 to 5 and print everything with f-strings.
Money uses the `,` format specifier (for example `{value:,}`), and column widths such as `<10` and `>12` keep the report aligned.

In [7]:
width = 52
line = "=" * width
thin = "-" * width

print(line)
print("STALL PERFORMANCE REPORT".center(width))
print("Wanjiku's Fresh Produce, Gikomba Market".center(width))
print(line)

print("\nSALES OVERVIEW (14 days)")
print(thin)
print(f"{'Total sales':<26}{'KES ' + format(total_sales, ','):>26}")
print(f"{'Average daily sales':<26}{'KES ' + format(average_sales, ',.2f'):>26}")
print(f"{'Best day':<26}{f'{best_day}, wk {best_week}: KES {highest_sales:,}':>26}")
print(f"{'Worst day':<26}{f'{worst_day}, wk {worst_week}: KES {lowest_sales:,}':>26}")

print("\nGROWTH")
print(thin)
print(f"{'Week 1 total':<26}{'KES ' + format(week1_total, ','):>26}")
print(f"{'Week 2 total':<26}{'KES ' + format(week2_total, ','):>26}")
print(f"{'Week-on-week growth':<26}{growth_pct:>25.1f}%")

print("\nDAY CLASSIFICATION")
print(thin)
for label in ("Strong", "Normal", "Weak"):
    print(f"{label + ' days':<26}{category_counts.get(label, 0):>26}")

print("\nWEEKDAY vs WEEKEND (average per day)")
print(thin)
print(f"{'Weekdays (Mon-Fri)':<26}{'KES ' + format(weekday_avg, ',.2f'):>26}")
print(f"{'Weekends (Sat-Sun)':<26}{'KES ' + format(weekend_avg, ',.2f'):>26}")
print(f"{'Weekend uplift':<26}{weekend_uplift_pct:>25.1f}%")

print("\nSTOCK & MARGINS")
print(thin)
print(f"{'Product':<12}{'Stock':>8}{'Reorder':>10}{'Margin':>9}{'Status':>13}")
for product, info in stock.items():
    status = "REORDER" if product in reorder_list else "OK"
    print(f"{product:<12}{info['qty']:>6}kg{info['reorder']:>8}kg{margins[product]:>8.1f}%{status:>13}")

print(f"\nRestock now: {', '.join(reorder_list)}")
print(line)

              STALL PERFORMANCE REPORT              
      Wanjiku's Fresh Produce, Gikomba Market       

SALES OVERVIEW (14 days)
----------------------------------------------------
Total sales                               KES 46,450
Average daily sales                     KES 3,317.86
Best day                        Sat, wk 2: KES 5,600
Worst day                       Tue, wk 1: KES 1,950

GROWTH
----------------------------------------------------
Week 1 total                              KES 22,050
Week 2 total                              KES 24,400
Week-on-week growth                            10.7%

DAY CLASSIFICATION
----------------------------------------------------
Strong days                                        4
Normal days                                        4
Weak days                                          6

WEEKDAY vs WEEKEND (average per day)
----------------------------------------------------
Weekdays (Mon-Fri)                      KES 2,685.00
Weekend

## Task 7: Business recommendations

**Approach:** I turned the results of Tasks 1 to 6 into five short recommendations. Each one is backed by a number from the analysis and tells Wanjiku what to do next.

## Recommendations for Wanjiku

1. **Your business is growing:** week 2 sales were KES 24,400 against KES 22,050 in week 1, a rise of 10.7% (KES 2,350), so keep doing what you did in week 2.
2. **Make weekends your priority:** Saturday and Sunday averaged KES 4,900 a day against KES 2,685 on weekdays (82.5% more), and those 4 days brought in KES 19,600, or 42.2% of all sales, so buy and display your best stock on Friday.
3. **Restock Tomatoes and Sukuma now:** Tomatoes are at 18 kg against a 25 kg reorder level (7 kg short) and Sukuma at 9 kg against 15 kg (6 kg short), while Onions (40 kg vs 20 kg) and Potatoes (55 kg vs 30 kg) are well stocked, so spend your next purchase on the first two instead.
4. **Protect Avocado, your most profitable product:** it earns a 42.9% gross margin (KES 60 profit on every kg), the highest on your stall, yet at 12 kg it is only 2 kg above its reorder level, so reorder it before it runs out.
5. **Fix the slow weekdays:** 6 of your 14 days were "Weak" (below KES 2,654), all of them Mondays, Tuesdays or Thursdays, and Tuesday averaged just KES 2,075, so try a small weekday offer (for example a bundle on Sukuma, which has a 40% margin) and buy less perishable stock for those days.